### Vector Store

#### ChromaDb and FAISS using LangChain

#### Addinf, updating and deleting information from/to ChromaDB

Embeddings from the da should be stored using a vector storem which can later retrieve info based on queries using similrity search.

ChromaDB and FAISS are a vector stores supported by langChain, that saves embeddings along with metadata

In [1]:
# All needed imports
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_2770541/1566530617.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
/root/opt/la-i-b/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Loading data file
loader = TextLoader("../../data/raw/txt/rag_notebook.txt")
data = loader.load()

# Splitting data
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    length_function=len,
)

chunks = text_splitter.split_documents(data)
chunks[0]

Document(metadata={'source': '../../data/raw/txt/rag_notebook.txt'}, page_content='- Consider these Moon Facts…')

In [3]:
# Hugging Face Embedding Model
model_name = "sentence-transformers/all-mpnet-base-v2"
huggingface_embedding = HuggingFaceEmbeddings(model_name=model_name)

query = "How are you?"
query_embedding = huggingface_embedding.embed_query(query)

/tmp/ipykernel_2770541/3584321757.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  huggingface_embedding = HuggingFaceEmbeddings(model_name=model_name)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8608.36it/s]


In [4]:
# Vector Database

# First, must create an ID list that will be used to assign each chunk a unique identifier, allowing us to track them later in the vector database. 
# The length of this list should match the length of the chunks.d
ids = [str(i) for i in range(0, len(chunks))]

vectordb = Chroma.from_documents(chunks, huggingface_embedding, ids=ids) # persistent_directory="./chroma_db", collection_name="rag_notebook")

In [5]:
for i in range(3):
    print(vectordb._collection.get(ids=str(i)))

vectordb._collection.count()

{'ids': ['0'], 'embeddings': None, 'documents': ['- Consider these Moon Facts…'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'source': '../../data/raw/txt/rag_notebook.txt'}]}
{'ids': ['1'], 'embeddings': None, 'documents': ['It takes 27 1/3 days for the Moon to go around Earth one time; called the Sidereal Month. From new'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'source': '../../data/raw/txt/rag_notebook.txt'}]}
{'ids': ['2'], 'embeddings': None, 'documents': ['Month. From new Moon to the next new Moon takes 29 1/2 days; called the Synodic Month. Why this'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'source': '../../data/raw/txt/rag_notebook.txt'}]}


28

In [6]:
# Similarity search
query = "What is young moon?"

docs = vectordb.similarity_search(query)
docs

[Document(id='22', metadata={'source': '../../data/raw/txt/rag_notebook.txt'}, page_content='Is there a Young Moon or an Old Moon? These are terms used to describe the Moon’s phase right'),
 Document(id='23', metadata={'source': '../../data/raw/txt/rag_notebook.txt'}, page_content='Moon’s phase right before (old) or after new moon (young). These are very thin crescent phases and'),
 Document(id='14', metadata={'source': '../../data/raw/txt/rag_notebook.txt'}, page_content='3. Waxing Moon – “Growing” larger; New Moon up to Full Moon'),
 Document(id='15', metadata={'source': '../../data/raw/txt/rag_notebook.txt'}, page_content='4. Waning Moon – “Growing” smaller; right past Full Moon to New Moon')]

------------

In [7]:
faissdb = FAISS.from_documents(chunks, huggingface_embedding, ids=ids)

for i in range(3):
    print(faissdb.docstore.search(str(i)))

page_content='- Consider these Moon Facts…' metadata={'source': '../../data/raw/txt/rag_notebook.txt'}
page_content='It takes 27 1/3 days for the Moon to go around Earth one time; called the Sidereal Month. From new' metadata={'source': '../../data/raw/txt/rag_notebook.txt'}
page_content='Month. From new Moon to the next new Moon takes 29 1/2 days; called the Synodic Month. Why this' metadata={'source': '../../data/raw/txt/rag_notebook.txt'}


In [8]:
# Similarity search
 
query = "What is young moon?"
docs = faissdb.similarity_search(query)
docs

[Document(id='22', metadata={'source': '../../data/raw/txt/rag_notebook.txt'}, page_content='Is there a Young Moon or an Old Moon? These are terms used to describe the Moon’s phase right'),
 Document(id='23', metadata={'source': '../../data/raw/txt/rag_notebook.txt'}, page_content='Moon’s phase right before (old) or after new moon (young). These are very thin crescent phases and'),
 Document(id='14', metadata={'source': '../../data/raw/txt/rag_notebook.txt'}, page_content='3. Waxing Moon – “Growing” larger; New Moon up to Full Moon'),
 Document(id='15', metadata={'source': '../../data/raw/txt/rag_notebook.txt'}, page_content='4. Waning Moon – “Growing” smaller; right past Full Moon to New Moon')]

--------

In [ ]:
# Adding entries from the vector store

text = "Moon is tidally locked to the Earth, meaning that the same side of the Moon always faces the Earth. This is due to the gravitational forces between the two bodies, which cause the Moon's rotation period to match its orbital period around the Earth."

new_chunk = Document(page_content=text, 
                    metadata={"source": "i heard it from AstroSock podcast mny times.", "page": 1})

new_chunks = [new_chunk]

# must find out he length of the new_chunks list and create a new ID list that will be used to assign each chunk a unique identifier, allowing us to track them later in the vector database.
vectordb._collection.count()

28

In [12]:
print(vectordb._collection.get(ids=['28']))

{'ids': [], 'embeddings': None, 'documents': [], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': []}


In [13]:
vectordb.add_documents(
    new_chunks,
    ids=["28"]
)
print(vectordb._collection.count())
print(vectordb._collection.get(ids=['28']))

29
{'ids': ['28'], 'embeddings': None, 'documents': ["Moon is tidally locked to the Earth, meaning that the same side of the Moon always faces the Earth. This is due to the gravitational forces between the two bodies, which cause the Moon's rotation period to match its orbital period around the Earth."], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'page': 1, 'source': 'i heard it from AstroSock podcast mny times.'}]}


In [ ]:
# Updating entries from the vector store
text = "Our beautiful Moon is tidally locked to the Earth, meaning that the same side of the Moon always faces the Earth. This is due to the gravitational forces between the two bodies, which cause the Moon's rotation period to match its orbital period around the Earth."

update_chunk = Document(page_content=text, metadata={"source": "Dr. Becky Smethurst", "page": 1})

vectordb.update_document('28', update_chunk,)
print(vectordb._collection.get(ids=['28']))

{'ids': ['28'], 'embeddings': None, 'documents': ["Our beautiful Moon is tidally locked to the Earth, meaning that the same side of the Moon always faces the Earth. This is due to the gravitational forces between the two bodies, which cause the Moon's rotation period to match its orbital period around the Earth."], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'page': 1, 'source': 'Dr. Becky Smethurst'}]}


In [16]:
# Deleting entries from the vector store 
vectordb._collection.delete(ids=['28'])
print(vectordb._collection.get(ids=['215']))

{'ids': [], 'embeddings': None, 'documents': [], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': []}
